# XGBoost — Base vs Google Trends

**Part 1** trains and evaluates XGBoost on price + engineered features.  
**Part 2** adds 5 Google Trends features and evaluates independently.  
**Part 3** compares both models head-to-head.

Both models use market-level sample weighting (`1 / n_snapshots`) and early stopping on a held-out validation set.

In [1]:
import itertools
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    f1_score, log_loss, precision_recall_curve, roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

In [2]:
ROOT            = pathlib.Path("../..")
DATA_DIR        = ROOT / "data"
ARTIFACTS_DIR   = pathlib.Path("artifacts")
PREDICTIONS_DIR = pathlib.Path("predictions")

NUMERIC_FEATURES = [
    "price_at_snapshot",
    "price_deviation_from_half",
    "days_before_close",
    "pct_lifetime_elapsed",
    "duration_days",
    "log_volume",
    "price_mean_7d",   "price_volatility_7d",  "price_min_7d",  "price_max_7d",
    "price_change_7d", "price_range_7d",        "price_trend_7d",
    "price_mean_14d",  "price_volatility_14d", "price_min_14d", "price_max_14d",
    "price_change_14d","price_range_14d",       "price_trend_14d",
]
TRENDS_FEATURES      = ["trend_value", "trend_ma4", "trend_change_4w", "trend_spike", "has_trend_data"]
CATEGORICAL_FEATURES = ["category"]
TARGET               = "outcome"

FEATURES_BASE   = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FEATURES_TRENDS = NUMERIC_FEATURES + TRENDS_FEATURES + CATEGORICAL_FEATURES

---
## Load Data

In [3]:
df = pd.read_parquet(DATA_DIR / "polymarket_ml_dataset_with_trends_clean.parquet")

df["category"] = df["category"].fillna("other")
df = df.dropna(subset=[TARGET])

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

assert len(set(train["market_id"]) & set(test["market_id"])) == 0, "Market leakage detected"

rng            = np.random.default_rng(42)
all_market_ids = train["market_id"].unique()

# 15% of markets for early stopping validation
val_ids = set(rng.choice(all_market_ids, size=int(len(all_market_ids) * 0.15), replace=False))

# 20% of remaining markets for isotonic calibration
remaining_ids = [m for m in all_market_ids if m not in val_ids]
cal_ids = set(rng.choice(remaining_ids, size=int(len(remaining_ids) * 0.20), replace=False))

train_fit = train[~train["market_id"].isin(val_ids) & ~train["market_id"].isin(cal_ids)]
train_val = train[ train["market_id"].isin(val_ids)]
train_cal = train[ train["market_id"].isin(cal_ids)]
test_reset = test.reset_index(drop=True)

y_fit  = train_fit[TARGET].values
y_val  = train_val[TARGET].values
y_cal  = train_cal[TARGET].values
y_test = test[TARGET].values

counts           = train_fit.groupby("market_id").size()
snapshot_weights = train_fit["market_id"].map(counts).rdiv(1).values
class_weights    = compute_sample_weight("balanced", y_fit)
sample_weights   = class_weights * snapshot_weights
sample_weights   = sample_weights / sample_weights.mean()

print(f"Total rows : {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Train fit  : {len(train_fit):,}  |  {train_fit['market_id'].nunique():,} markets")
print(f"Train val  : {len(train_val):,}   |  {train_val['market_id'].nunique():,} markets (early stopping)")
print(f"Train cal  : {len(train_cal):,}   |  {train_cal['market_id'].nunique():,} markets (isotonic calibration)")
print(f"Test       : {len(test):,}   |  {test['market_id'].nunique():,} markets")
df.head()

Total rows : 1,448,142  |  Columns: 32
Train fit  : 790,479  |  11,407 markets
Train val  : 171,723   |  2,516 markets (early stopping)
Train cal  : 197,450   |  2,851 markets (isotonic calibration)
Test       : 288,490   |  4,174 markets


,market_id,snapshot_timestamp,days_before_close,pct_lifetime_elapsed,duration_days,price_at_snapshot,price_deviation_from_half,total_volume,log_volume,outcome,...,price_range_14d,price_trend_14d,split,category,question,trend_value,trend_ma4,trend_change_4w,trend_spike,has_trend_data
0,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-01 18:45:42.437000+00:00,59.22,0.1912,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000526,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
1,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 06:45:42.437000+00:00,58.72,0.1980,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000354,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
2,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-02 18:45:42.437000+00:00,58.22,0.2049,73,0.03,0.47,40175.18,10.601,0,...,0.05,0.000215,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
3,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 06:45:42.437000+00:00,57.72,0.2117,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000479,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1
4,0xa4327fa7fc306ac5bcbb7ad564486b2bc303e37afff4...,2023-03-03 18:45:42.437000+00:00,57.22,0.2185,73,0.03,0.47,40175.18,10.601,0,...,0.03,0.000499,train,crypto,Will USDC redemption or minting be halted in t...,8.8,9.7,-2.0,0,1


---
## Shared Helpers

In [4]:
def build_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="constant", fill_value=0), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

def build_xgb(learning_rate=0.05, max_depth=6, min_child_weight=10):
    return XGBClassifier(
        n_estimators=1000,
        early_stopping_rounds=50,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=min_child_weight,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

def get_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thresh[np.argmax(f1)]), float(np.max(f1))

def evaluate(y_true, y_prob, label, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "AUC-ROC" : roc_auc_score(y_true, y_prob),
        "PR-AUC"  : average_precision_score(y_true, y_prob),
        "Log-loss": log_loss(y_true, y_prob),
        "Brier"   : brier_score_loss(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1"      : f1_score(y_true, y_pred),
    }
    print(f"\n{'─'*50}")
    print(f"  {label}  (threshold={threshold:.3f})")
    print(f"{'─'*50}")
    for name, val in metrics.items():
        print(f"  {name:<10}: {val:.4f}")
    print(f"{'─'*50}")
    return metrics

def per_category(test_df, y_prob, threshold):
    rows = []
    for cat in sorted(test_df["category"].unique()):
        mask = test_df["category"] == cat
        yt   = test_df.loc[mask, TARGET].values
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            continue
        yp = y_prob[mask.values]
        rows.append({
            "category": cat,
            "n"       : int(mask.sum()),
            "AUC"     : roc_auc_score(yt, yp),
            "PR-AUC"  : average_precision_score(yt, yp),
            "F1"      : f1_score(yt, (yp >= threshold).astype(int)),
            "YES%"    : float(yt.mean()),
        })
    return pd.DataFrame(rows).set_index("category").sort_values("AUC", ascending=False)

def market_eval(test_df, y_prob, threshold):
    mdf = (
        test_df.assign(pred_prob=y_prob)
        .groupby("market_id")
        .agg(pred_prob=("pred_prob", "mean"), outcome=(TARGET, "first"))
        .reset_index()
    )
    mp, mt = mdf["pred_prob"].values, mdf["outcome"].values
    mpred  = (mp >= threshold).astype(int)
    print(f"Market-level evaluation ({len(mdf):,} markets)")
    print(f"  AUC-ROC  : {roc_auc_score(mt, mp):.4f}")
    print(f"  PR-AUC   : {average_precision_score(mt, mp):.4f}")
    print(f"  Brier    : {brier_score_loss(mt, mp):.4f}")
    print(f"  Accuracy : {accuracy_score(mt, mpred):.4f}")
    print(f"  F1       : {f1_score(mt, mpred):.4f}")
    return {
        "AUC-ROC" : roc_auc_score(mt, mp),
        "PR-AUC"  : average_precision_score(mt, mp),
        "Brier"   : brier_score_loss(mt, mp),
        "Accuracy": accuracy_score(mt, mpred),
        "F1"      : f1_score(mt, mpred),
    }

def feature_importance_df(preprocessor, clf, num_cols, trend_cols=None):
    cat_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
    all_names = num_cols + cat_names
    return (
        pd.DataFrame({"feature": all_names, "importance": clf.feature_importances_})
        .assign(is_trend=lambda d: d["feature"].isin(trend_cols or []))
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

---
## Grid Search — Hyperparameter Tuning

Search over `learning_rate` and `max_depth` (9 combinations, `min_child_weight` fixed at 10).
Early stopping on the validation split determines `n_estimators` for each. Best params are used for both Base and Trends models.

In [5]:
_param_grid = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth":     [4, 6, 8],
}
_combinations = list(itertools.product(
    _param_grid["learning_rate"],
    _param_grid["max_depth"],
))

_prep_gs  = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
_X_fit_gs = _prep_gs.fit_transform(train_fit[FEATURES_BASE])
_X_val_gs = _prep_gs.transform(train_val[FEATURES_BASE])

print(f"Grid search: {len(_combinations)} combinations...")
_best_score, _best_params = float("inf"), None

for i, (lr, depth) in enumerate(_combinations, 1):
    _clf = build_xgb(learning_rate=lr, max_depth=depth)
    _clf.fit(_X_fit_gs, y_fit, sample_weight=sample_weights, eval_set=[(_X_val_gs, y_val)], verbose=False)
    score = _clf.best_score
    print(f"  [{i}/{len(_combinations)}]  lr={lr}  depth={depth}  \u2192  logloss={score:.4f}  (iters={_clf.best_iteration})")
    if score < _best_score:
        _best_score  = score
        _best_params = {"learning_rate": lr, "max_depth": depth}

print(f"\nBest params : {_best_params}")
print(f"Best logloss: {_best_score:.4f}")

Grid search: 9 combinations...
  [1/9]  lr=0.01  depth=4  →  logloss=0.3660  (iters=999)
  [2/9]  lr=0.01  depth=6  →  logloss=0.3574  (iters=999)
  [3/9]  lr=0.01  depth=8  →  logloss=0.3451  (iters=998)
  [4/9]  lr=0.05  depth=4  →  logloss=0.3504  (iters=866)
  [5/9]  lr=0.05  depth=6  →  logloss=0.3441  (iters=600)
  [6/9]  lr=0.05  depth=8  →  logloss=0.3394  (iters=403)
  [7/9]  lr=0.1  depth=4  →  logloss=0.3565  (iters=330)
  [8/9]  lr=0.1  depth=6  →  logloss=0.3449  (iters=329)
  [9/9]  lr=0.1  depth=8  →  logloss=0.3464  (iters=139)

Best params : {'learning_rate': 0.05, 'max_depth': 8}
Best logloss: 0.3394


---
## Baseline — Market Price

The simplest predictor: use `price_at_snapshot` directly as the probability.
This is the crowd's consensus — a useful sanity check for any model we build.

In [6]:
y_prob_baseline        = test_reset["price_at_snapshot"].values
thresh_bl, f1_bl       = get_threshold(y_test, y_prob_baseline)
print(f"Baseline threshold: {thresh_bl:.3f}  |  F1: {f1_bl:.4f}")

metrics_baseline_row = evaluate(y_test, y_prob_baseline, "Baseline (market price)", thresh_bl)
metrics_baseline_mkt = market_eval(test_reset, y_prob_baseline, thresh_bl)
cat_baseline         = per_category(test_reset, y_prob_baseline, thresh_bl)

Baseline threshold: 0.500  |  F1: 0.6699

──────────────────────────────────────────────────
  Baseline (market price)  (threshold=0.500)
──────────────────────────────────────────────────
  AUC-ROC   : 0.8890
  PR-AUC    : 0.7440
  Log-loss  : 0.3278
  Brier     : 0.1004
  Accuracy  : 0.8703
  F1        : 0.6699
──────────────────────────────────────────────────
Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9420
  PR-AUC   : 0.8283
  Brier    : 0.0652
  Accuracy : 0.9178
  F1       : 0.7351


---
# Part 1 — Base Model

Price + engineered features only, no Google Trends.

## 1.1 Train

In [7]:
prep_base   = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
X_fit_base  = prep_base.fit_transform(train_fit[FEATURES_BASE])
X_val_base  = prep_base.transform(train_val[FEATURES_BASE])
X_cal_base  = prep_base.transform(train_cal[FEATURES_BASE])
X_test_base = prep_base.transform(test[FEATURES_BASE])

clf_base = build_xgb(**_best_params)
clf_base.fit(
    X_fit_base, y_fit,
    sample_weight=sample_weights,
    eval_set=[(X_val_base, y_val)],
    verbose=False,
)
print(f"Best iteration: {clf_base.best_iteration}  |  Best logloss: {clf_base.best_score:.4f}")

# Isotonic calibration on held-out calibration set
p_cal_raw_base = clf_base.predict_proba(X_cal_base)[:, 1]
iso_base = IsotonicRegression(out_of_bounds="clip")
iso_base.fit(p_cal_raw_base, y_cal)

y_prob_base          = iso_base.transform(clf_base.predict_proba(X_test_base)[:, 1])
thresh_base, f1_base = get_threshold(y_test, y_prob_base)
print(f"Optimal threshold: {thresh_base:.3f}  |  F1: {f1_base:.4f}")

Best iteration: 403  |  Best logloss: 0.3394
Optimal threshold: 0.317  |  F1: 0.6728


## 1.2 Row-Level Evaluation

In [8]:
metrics_base_row = evaluate(y_test, y_prob_base, "XGBoost Base", thresh_base)


──────────────────────────────────────────────────
  XGBoost Base  (threshold=0.317)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9000
  PR-AUC    : 0.7445
  Log-loss  : 0.3072
  Brier     : 0.0941
  Accuracy  : 0.8462
  F1        : 0.6728
──────────────────────────────────────────────────


## 1.3 Per-Category Breakdown

In [9]:
cat_base = per_category(test_reset, y_prob_base, thresh_base)
cat_base.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
crypto,24586,0.9341,0.8397,0.8003,26.1%
entertainment,37027,0.9292,0.7375,0.6288,14.9%
geopolitics,16578,0.9229,0.7137,0.6048,13.8%
politics_us,52699,0.9155,0.8143,0.7106,24.7%
politics_global,22591,0.9091,0.7418,0.6829,22.3%
finance,25093,0.8977,0.7845,0.6864,24.3%
science_tech,15530,0.8910,0.7077,0.6296,17.8%
other,1364,0.8778,0.3869,0.4302,4.7%
sports,93022,0.8696,0.6750,0.6304,20.9%


## 1.4 Market-Level Evaluation

In [10]:
metrics_base_mkt = market_eval(test_reset, y_prob_base, thresh_base)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9405
  PR-AUC   : 0.8220
  Brier    : 0.0642
  Accuracy : 0.9034
  F1       : 0.7350


## 1.5 Feature Importance

In [11]:
imp_base = feature_importance_df(prep_base, clf_base, NUMERIC_FEATURES)
imp_base.head(20).style.bar(subset=["importance"], color="#5fba7d").format({"importance": "{:.4f}"})

,feature,importance,is_trend
0,price_at_snapshot,0.4669,False
1,price_mean_7d,0.1270,False
2,price_deviation_from_half,0.0640,False
3,price_max_7d,0.0245,False
4,log_volume,0.0200,False
5,category_geopolitics,0.0195,False
6,category_other,0.0181,False
7,duration_days,0.0173,False
8,category_sports,0.0170,False
9,category_politics_us,0.0166,False


---
# Part 2 — Model with Google Trends

Same as Part 1 plus 5 Google Trends features: `trend_value`, `trend_ma4`, `trend_change_4w`, `trend_spike`, `has_trend_data`.

## 2.1 Train

In [12]:
prep_trends   = build_preprocessor(NUMERIC_FEATURES + TRENDS_FEATURES, CATEGORICAL_FEATURES)
X_fit_trends  = prep_trends.fit_transform(train_fit[FEATURES_TRENDS])
X_val_trends  = prep_trends.transform(train_val[FEATURES_TRENDS])
X_cal_trends  = prep_trends.transform(train_cal[FEATURES_TRENDS])
X_test_trends = prep_trends.transform(test[FEATURES_TRENDS])

clf_trends = build_xgb(**_best_params)
clf_trends.fit(
    X_fit_trends, y_fit,
    sample_weight=sample_weights,
    eval_set=[(X_val_trends, y_val)],
    verbose=False,
)
print(f"Best iteration: {clf_trends.best_iteration}  |  Best logloss: {clf_trends.best_score:.4f}")

p_cal_raw_trends = clf_trends.predict_proba(X_cal_trends)[:, 1]
iso_trends = IsotonicRegression(out_of_bounds="clip")
iso_trends.fit(p_cal_raw_trends, y_cal)

y_prob_trends            = iso_trends.transform(clf_trends.predict_proba(X_test_trends)[:, 1])
thresh_trends, f1_trends = get_threshold(y_test, y_prob_trends)
print(f"Optimal threshold: {thresh_trends:.3f}  |  F1: {f1_trends:.4f}")

Best iteration: 388  |  Best logloss: 0.3359
Optimal threshold: 0.348  |  F1: 0.6733


## 2.2 Row-Level Evaluation

In [13]:
metrics_trends_row = evaluate(y_test, y_prob_trends, "XGBoost Trends", thresh_trends)


──────────────────────────────────────────────────
  XGBoost Trends  (threshold=0.348)
──────────────────────────────────────────────────
  AUC-ROC   : 0.9019
  PR-AUC    : 0.7525
  Log-loss  : 0.3043
  Brier     : 0.0930
  Accuracy  : 0.8476
  F1        : 0.6733
──────────────────────────────────────────────────


## 2.3 Per-Category Breakdown

In [14]:
cat_trends = per_category(test_reset, y_prob_trends, thresh_trends)
cat_trends.style.format({"AUC": "{:.4f}", "PR-AUC": "{:.4f}", "F1": "{:.4f}", "YES%": "{:.1%}"})

,n,AUC,PR-AUC,F1,YES%
category,,,,,
geopolitics,16578,0.9354,0.7479,0.6467,13.8%
crypto,24586,0.9291,0.8280,0.7767,26.1%
entertainment,37027,0.9277,0.7173,0.6492,14.9%
politics_us,52699,0.9159,0.8211,0.7180,24.7%
politics_global,22591,0.9060,0.7471,0.6896,22.3%
finance,25093,0.9039,0.7955,0.6919,24.3%
science_tech,15530,0.8868,0.6982,0.6160,17.8%
sports,93022,0.8727,0.6917,0.6193,20.9%
other,1364,0.8122,0.1760,0.2879,4.7%


## 2.4 Market-Level Evaluation

In [15]:
metrics_trends_mkt = market_eval(test_reset, y_prob_trends, thresh_trends)

Market-level evaluation (4,174 markets)
  AUC-ROC  : 0.9431
  PR-AUC   : 0.8298
  Brier    : 0.0627
  Accuracy : 0.9073
  F1       : 0.7418


## 2.5 Feature Importance

Trend features are highlighted in yellow.

In [16]:
imp_trends = feature_importance_df(prep_trends, clf_trends, NUMERIC_FEATURES + TRENDS_FEATURES, TRENDS_FEATURES)

print("Trend feature importances:")
print(imp_trends[imp_trends["is_trend"]].to_string(index=False))
print()

imp_trends.head(25).style.bar(
    subset=["importance"], color="#5fba7d"
).apply(
    lambda col: ["background-color: #fff3cd" if v else "" for v in imp_trends.head(25)["is_trend"]],
    axis=0, subset=["feature", "importance"]
).format({"importance": "{:.4f}"})

Trend feature importances:
        feature  importance  is_trend
 has_trend_data    0.012935      True
      trend_ma4    0.012325      True
    trend_value    0.011675      True
    trend_spike    0.009819      True
trend_change_4w    0.009791      True



,feature,importance,is_trend
0,price_at_snapshot,0.3746,False
1,price_mean_7d,0.1833,False
2,price_deviation_from_half,0.0530,False
3,price_min_7d,0.0466,False
4,log_volume,0.0178,False
5,category_geopolitics,0.0162,False
6,category_sports,0.0155,False
7,duration_days,0.0150,False
8,price_max_7d,0.0140,False
9,category_politics_global,0.0140,False


---
# Part 3 — Comparison

Head-to-head: Base vs Trends across row-level metrics, market-level metrics, and per-category AUC.

## 3.1 Row-Level

In [17]:
row_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_row,
    "Base":     metrics_base_row,
    "Trends":   metrics_trends_row,
})
row_comparison["\u0394 Base"]   = row_comparison["Base"]   - row_comparison["Baseline"]
row_comparison["\u0394 Trends"] = row_comparison["Trends"] - row_comparison["Baseline"]
row_comparison.style.format("{:.4f}").bar(
    subset=["\u0394 Base", "\u0394 Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.8890,0.9000,0.9019,0.0110,0.0129
PR-AUC,0.7440,0.7445,0.7525,0.0005,0.0085
Log-loss,0.3278,0.3072,0.3043,-0.0206,-0.0236
Brier,0.1004,0.0941,0.0930,-0.0063,-0.0074
Accuracy,0.8703,0.8462,0.8476,-0.0241,-0.0228
F1,0.6699,0.6728,0.6733,0.0030,0.0034


## 3.2 Market-Level

In [18]:
mkt_comparison = pd.DataFrame({
    "Baseline": metrics_baseline_mkt,
    "Base":     metrics_base_mkt,
    "Trends":   metrics_trends_mkt,
})
mkt_comparison["\u0394 Base"]   = mkt_comparison["Base"]   - mkt_comparison["Baseline"]
mkt_comparison["\u0394 Trends"] = mkt_comparison["Trends"] - mkt_comparison["Baseline"]
mkt_comparison.style.format("{:.4f}").bar(
    subset=["\u0394 Base", "\u0394 Trends"], align="zero", color=["#d65f5f", "#5fba7d"]
)

,Baseline,Base,Trends,Δ Base,Δ Trends
AUC-ROC,0.9420,0.9405,0.9431,-0.0015,0.0011
PR-AUC,0.8283,0.8220,0.8298,-0.0063,0.0015
Brier,0.0652,0.0642,0.0627,-0.0010,-0.0025
Accuracy,0.9178,0.9034,0.9073,-0.0144,-0.0105
F1,0.7351,0.7350,0.7418,-0.0001,0.0067


## 3.3 Per-Category AUC Delta

In [19]:
cat_comparison = cat_baseline[["n", "AUC", "YES%"]].rename(columns={"AUC": "AUC (baseline)"})
cat_comparison["AUC (base)"]   = cat_base["AUC"]
cat_comparison["AUC (trends)"] = cat_trends["AUC"]
cat_comparison["\u0394 Base"]       = cat_comparison["AUC (base)"]   - cat_comparison["AUC (baseline)"]
cat_comparison["\u0394 Trends"]     = cat_comparison["AUC (trends)"] - cat_comparison["AUC (baseline)"]
(
    cat_comparison
    .sort_values("AUC (baseline)", ascending=False)
    .style
    .format("{:.4f}", subset=["AUC (baseline)", "AUC (base)", "AUC (trends)", "\u0394 Base", "\u0394 Trends"])
    .format("{:.1%}", subset=["YES%"])
    .bar(subset=["\u0394 Base", "\u0394 Trends"], align="zero", color=["#d65f5f", "#5fba7d"])
)

,n,AUC (baseline),YES%,AUC (base),AUC (trends),Δ Base,Δ Trends
category,,,,,,,
geopolitics,16578,0.9265,13.8%,0.9229,0.9354,-0.0036,0.0089
finance,25093,0.9183,24.3%,0.8977,0.9039,-0.0206,-0.0144
entertainment,37027,0.9101,14.9%,0.9292,0.9277,0.0191,0.0176
politics_us,52699,0.9083,24.7%,0.9155,0.9159,0.0071,0.0076
politics_global,22591,0.9057,22.3%,0.9091,0.9060,0.0034,0.0003
crypto,24586,0.9052,26.1%,0.9341,0.9291,0.0289,0.0239
science_tech,15530,0.8876,17.8%,0.8910,0.8868,0.0034,-0.0009
other,1364,0.8847,4.7%,0.8778,0.8122,-0.0069,-0.0725
sports,93022,0.8400,20.9%,0.8696,0.8727,0.0296,0.0327


---
## Save Artifacts

In [22]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({"preprocessor": prep_base,   "clf": clf_base,   "calibrator": iso_base},   ARTIFACTS_DIR / "model_base.joblib")
joblib.dump({"preprocessor": prep_trends, "clf": clf_trends, "calibrator": iso_trends}, ARTIFACTS_DIR / "model_trends.joblib")
print(f"Models saved → {ARTIFACTS_DIR}")

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
pred_df = test[["market_id", "snapshot_timestamp", "category", TARGET]].copy().reset_index(drop=True)
pred_df["pred_prob_base"]    = y_prob_base
pred_df["pred_prob_trends"]  = y_prob_trends
pred_df["pred_label_base"]   = (y_prob_base   >= thresh_base).astype(int)
pred_df["pred_label_trends"] = (y_prob_trends >= thresh_trends).astype(int)
preds_path = PREDICTIONS_DIR / "predictions.csv"
pred_df.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")
pred_df.head()

Models saved → artifacts
Predictions saved → predictions/predictions.csv


,market_id,snapshot_timestamp,category,outcome,pred_prob_base,pred_prob_trends,pred_label_base,pred_label_trends
0,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-09 21:08:18.864000+00:00,crypto,0,0.328144,0.466419,1,1
1,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-10 09:08:18.864000+00:00,crypto,0,0.448458,0.452865,1,1
2,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-10 21:08:18.864000+00:00,crypto,0,0.505573,0.488789,1,1
3,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-11 09:08:18.864000+00:00,crypto,0,0.430622,0.466419,1,1
4,0xf8644ffe668cfec9044c2b64f9825ffce8520b521df5...,2023-03-11 21:08:18.864000+00:00,crypto,0,0.541425,0.588329,1,1
